#LAB TUTORIAL 7
Second-Order Optimization Methods

###0. Setup

In [2]:
import time
import torch
import torch.nn as nn
torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)
print("PyTorch:", torch.__version__)

PyTorch: 2.11.0+cpu


###1. Gradient, Hessian and the Newton step

In [4]:
def L(w):
  return w**2 - 4*w + 5
w = torch.tensor(5.0, requires_grad=True)
g = torch.autograd.grad(L(w), w, create_graph=True)[0] # keep the graph alive
H = torch.autograd.grad(g, w)[0] # differentiate again
w_new = w - g / H
print(f"w0 = {w.item():.4f} g = {g.item():.4f} H = {H.item():.4f}")
print(f"w1 = w0 - g/H = {w_new.item():.4f}")
print(f"L(w0) = {L(w).item():.4f} -> L(w1) = {L(w_new).item():.4f}")

w0 = 5.0000 g = 6.0000 H = 2.0000
w1 = w0 - g/H = 2.0000
L(w0) = 10.0000 -> L(w1) = 1.0000


###2. Newton versus gradient descent on a stretched valley

In [6]:
from torch.autograd.functional import hessian
A = torch.tensor([[20.0, 0.0], [0.0, 1.0]])
b = torch.tensor([2.0, 1.0])
def loss(w):
  return 0.5 * w @ A @ w - b @ w
w_star = torch.linalg.solve(A, b)
print("optimum :", w_star)
w = torch.tensor([3.0, -4.0], requires_grad=True)
for i in range(1, 61):
  gr = torch.autograd.grad(loss(w), w)[0]
  with torch.no_grad():
    w -= 0.09 * gr # largest stable step is 2/20 = 0.1
  if i in (1, 10, 30, 60):
    print(f"GD iter {i:3d} error = {(w - w_star).norm().item():.6f}")
w = torch.tensor([3.0, -4.0], requires_grad=True)
gr = torch.autograd.grad(loss(w), w)[0]
Hm = hessian(loss, w)
w_newton = w.detach() - torch.linalg.solve(Hm, gr)
print("Hessian :", Hm.tolist())
print(f"Newton 1 step error = {(w_newton - w_star).norm().item():.6e}")

optimum : tensor([0.1000, 1.0000])
GD iter   1 error = 5.107338
GD iter  10 error = 1.971822
GD iter  30 error = 0.295287
GD iter  60 error = 0.017436
Hessian : [[20.0, 0.0], [0.0, 1.0]]
Newton 1 step error = 9.685755e-08


###3. What the Hessian actually costs

In [11]:
from torch.func import functional_call
from torch.autograd.functional import hessian
def hessian_cost(hidden):
  torch.manual_seed(0)
  net = nn.Sequential(nn.Linear(4, hidden), nn.Tanh(), nn.Linear(hidden, 1))
  names = [n for n, _ in net.named_parameters()]
  flat = torch.cat([p.reshape(-1) for p in net.parameters()])
  x, y = torch.randn(16, 4), torch.randn(16, 1)
  def flat_loss(v):
    params, i = {}, 0
    for n, p in zip(names, net.parameters()):
      params[n] = v[i:i + p.numel()].view_as(p)
      i += p.numel()
    return ((functional_call(net, params, (x,)) - y) ** 2).mean()
  t0 = time.time()
  Hm = hessian(flat_loss, flat)
  dt = time.time() - t0
  return flat.numel(), dt, Hm.numel() * 4 / 1024**2
hessian_cost(4) # warm-up, result discarded
print(f"{'params n':>9}{'Hessian time (s)':>19}{'Hessian memory (MB)':>22}")
for hidden in (8, 32, 128, 512):
  n, dt, mb = hessian_cost(hidden)
  print(f"{n:9d}{dt:19.3f}{mb:22.2f}")
n = 10_000_000 # a small real network
print(f"extrapolated to n = {n:,}: {n**2 * 4 / 1024**4:,.0f} TB")

 params n   Hessian time (s)   Hessian memory (MB)
       49              0.027                  0.01
      193              0.105                  0.14
      769              0.419                  2.26
     3073              1.914                 36.02
extrapolated to n = 10,000,000: 364 TB


###4. Gauss–Newton and Levenberg–Marquardt

In [12]:
import torch
from torch.autograd.functional import jacobian

torch.manual_seed(1)  # so this task stands alone

x = torch.linspace(0, 2, 40)

y = 3.0 * torch.exp(-1.5 * x) + 0.02 * torch.randn(40)


def residual(theta):
    return theta[0] * torch.exp(theta[1] * x) - y


def lm(lmbda, iters=10):
    theta = torch.tensor([1.0, -0.2])

    for _ in range(iters):
        r = residual(theta)
        J = jacobian(residual, theta)  # 40 x 2

        JtJ = J.T @ J

        step = torch.linalg.solve(
            JtJ + lmbda * torch.eye(2),
            J.T @ r
        )

        theta = theta - step

    return theta, (residual(theta) ** 2).sum().item()


print(
    f"{'method':<22}"
    f"{'SSE after 3':>14}"
    f"{'SSE after 10':>14} theta after 10"
)

for lmbda, name in [
    (0.0, "Gauss-Newton (lam=0)"),
    (1.0, "LM (lam=1)"),
    (1000.0, "LM (lam=1000)")
]:
    _, sse3 = lm(lmbda, iters=3)
    th, sse10 = lm(lmbda, iters=10)

    print(
        f"{name:<22}"
        f"{sse3:14.5f}"
        f"{sse10:14.5f}"
        f" [{th[0]:.4f}, {th[1]:.4f}]"
    )

method                   SSE after 3  SSE after 10 theta after 10
Gauss-Newton (lam=0)         0.01275       0.01275 [2.9702, -1.4856]
LM (lam=1)                   0.02001       0.01275 [2.9702, -1.4856]
LM (lam=1000)               19.82258      18.62383 [1.0708, -0.2604]


###5. L-BFGS: quasi-Newton in one line

In [17]:
def fit(make_opt, steps):
  torch.manual_seed(0)
  theta = torch.tensor([1.0, -0.2], requires_grad=True)
  opt = make_opt([theta])

  def closure(): # L-BFGS may call this many times
    opt.zero_grad()
    loss = (residual(theta) ** 2).sum()
    loss.backward()
    return loss

  for _ in range(steps):
    opt.step(closure)

  return theta.detach(), (residual(theta) ** 2).sum().item()

for steps in (10, 50, 200, 500):
  _, sse = fit(lambda p: torch.optim.Adam(p, lr=0.05), steps)
  print(f"Adam {steps:4d} steps SSE = {sse:.5f}")

for steps in (1, 2, 5):
  th, sse = fit(lambda p: torch.optim.LBFGS(p, lr=0.5, max_iter=20), steps)
  print(f"L-BFGS {steps:4d} steps SSE = {sse:.5f}")

Adam   10 steps SSE = 9.88366
Adam   50 steps SSE = 0.07337
Adam  200 steps SSE = 0.01275
Adam  500 steps SSE = 0.01275
L-BFGS    1 steps SSE = 0.01275
L-BFGS    2 steps SSE = 0.01275
L-BFGS    5 steps SSE = 0.01275


###6. Hessian-free optimization

In [19]:
def hvp(f, w, v):
  g = torch.autograd.grad(f(w), w, create_graph=True)[0]
  return torch.autograd.grad(g @ v, w)[0] # H @ v, H never built
def conjugate_gradient(f, w, g, iters=10, tol=1e-10):
  d = torch.zeros_like(g)
  r = -g.clone()
  p = r.clone()
  for k in range(iters):
    Hp = hvp(f, w, p)
    alpha = (r @ r) / (p @ Hp)
    d = d + alpha * p
    r_new = r - alpha * Hp
    print(f" CG iter {k+1}: residual = {r_new.norm().item():.3e}")
    if r_new.norm() < tol:
      break
    p = r_new + (r_new @ r_new) / (r @ r) * p
    r = r_new
  return d
w = torch.tensor([3.0, -4.0], requires_grad=True)
g = torch.autograd.grad(loss(w), w, create_graph=True)[0]
d = conjugate_gradient(loss, w, g.detach())
print("CG direction :", d)
print("exact -H^-1 g :", -torch.linalg.solve(A, g.detach()))
print("w after step :", (w.detach() + d))

 CG iter 1: residual = 4.766e+00
 CG iter 2: residual = 3.007e-05
 CG iter 3: residual = 9.518e-12
CG direction : tensor([-2.9000,  5.0000])
exact -H^-1 g : tensor([-2.9000,  5.0000])
w after step : tensor([0.1000, 1.0000])


##8. Lab exercises


### Question 1: Apply the Task 1 code to L(w) = 3w² − 12w + 7 starting from w = 4. Report g, H and w₁, and confirm by hand that the method reaches the optimum in one step.

In [20]:
# Define the new loss function L(w)
def L_q1(w):
  return 3*w**2 - 12*w + 7

# Initialize w to 4
w_q1 = torch.tensor(4.0, requires_grad=True)

# Compute gradient g
g_q1 = torch.autograd.grad(L_q1(w_q1), w_q1, create_graph=True)[0]

# Compute Hessian H
H_q1 = torch.autograd.grad(g_q1, w_q1)[0]

# Compute new w using Newton's step
w_new_q1 = w_q1 - g_q1 / H_q1

print(f"Initial w = {w_q1.item():.4f}")
print(f"Gradient g = {g_q1.item():.4f}")
print(f"Hessian H = {H_q1.item():.4f}")
print(f"New w (w₁) = {w_new_q1.item():.4f}")
print(f"L(initial w) = {L_q1(w_q1).item():.4f}")
print(f"L(new w) = {L_q1(w_new_q1).item():.4f}")

# Confirm by hand:
# L'(w) = 6w - 12
# Setting L'(w) = 0 gives 6w = 12, so w = 2.
# The Newton step from w=4 should directly go to w=2.
# Our calculation shows w₁ = 2.0000, which confirms it reaches the optimum in one step.

Initial w = 4.0000
Gradient g = 12.0000
Hessian H = 6.0000
New w (w₁) = 2.0000
L(initial w) = 7.0000
L(new w) = -5.0000


### Question 2: Change A in Task 2 to [[20, 0], [0, 0.05]] and rerun. How many gradient-descent iterations are now needed to match one Newton step, and how does that number relate to the condition number of A?

In [26]:
from torch.autograd.functional import hessian

# Define the new A matrix
A_q2 = torch.tensor([[20.0, 0.0], [0.0, 0.05]])
b_q2 = torch.tensor([2.0, 1.0])

def loss_q2(w):
  return 0.5 * w @ A_q2 @ w - b_q2 @ w

w_star_q2 = torch.linalg.solve(A_q2, b_q2)
print("Optimal w_star :", w_star_q2)

# --- Gradient Descent with new A ---
print("\n--- Gradient Descent ---")
w_gd_q2 = torch.tensor([3.0, -4.0], requires_grad=True)

# Learning rate for GD (largest stable step for A=[[20,0],[0,0.05]] is 2/20 = 0.1)
# For a general quadratic, the optimal learning rate is 2 / (lambda_max + lambda_min)
# Here, lambda_max = 20, lambda_min = 0.05. So 2 / (20 + 0.05) approx 0.099
learning_rate_q2 = 0.099

gd_errors = []
for i in range(1, 20001): # Increased iterations to observe convergence
  gr_gd_q2 = torch.autograd.grad(loss_q2(w_gd_q2), w_gd_q2)[0]
  with torch.no_grad():
    w_gd_q2 -= learning_rate_q2 * gr_gd_q2
  error_gd = (w_gd_q2 - w_star_q2).norm().item()
  gd_errors.append(error_gd)
  if i % 1000 == 0 or i == 1:
    print(f"GD iter {i:5d} error = {error_gd:.6e}")

# --- Newton's Method with new A ---
print("\n--- Newton's Method ---")
w_newton_q2 = torch.tensor([3.0, -4.0], requires_grad=True)
gr_newton_q2 = torch.autograd.grad(loss_q2(w_newton_q2), w_newton_q2)[0]
Hm_newton_q2 = hessian(loss_q2, w_newton_q2)
w_newton_step_q2 = w_newton_q2.detach() - torch.linalg.solve(Hm_newton_q2, gr_newton_q2)
error_newton = (w_newton_step_q2 - w_star_q2).norm().item()
print(f"Newton 1 step error = {error_newton:.6e}")

# --- Comparison and Condition Number ---
print("\n--- Analysis ---")
for i, err in enumerate(gd_errors):
  if err <= error_newton:
    print(f"Gradient Descent takes approximately {i+1} iterations to achieve Newton's error.")
    break

# Condition number of A
eigenvalues_A_q2 = torch.linalg.eigvalsh(A_q2)
condition_number_A_q2 = torch.max(eigenvalues_A_q2) / torch.min(eigenvalues_A_q2)
print(f"Condition number of A: {condition_number_A_q2:.2f}")
print("\nExplanation: A higher condition number indicates a more ill-conditioned problem (a 'stretched valley'), which makes Gradient Descent converge much slower. The large condition number of 400 makes GD require many more iterations to reach the precision of Newton's method.")

Optimal w_star : tensor([ 0.1000, 20.0000])

--- Gradient Descent ---
GD iter     1 error = 2.404971e+01
GD iter  1000 error = 1.679287e-01
GD iter  2000 error = 1.186371e-03
GD iter  3000 error = 1.926428e-04
GD iter  4000 error = 1.926428e-04
GD iter  5000 error = 1.926428e-04
GD iter  6000 error = 1.926428e-04
GD iter  7000 error = 1.926428e-04
GD iter  8000 error = 1.926428e-04
GD iter  9000 error = 1.926428e-04
GD iter 10000 error = 1.926428e-04
GD iter 11000 error = 1.926428e-04
GD iter 12000 error = 1.926428e-04
GD iter 13000 error = 1.926428e-04
GD iter 14000 error = 1.926428e-04
GD iter 15000 error = 1.926428e-04
GD iter 16000 error = 1.926428e-04
GD iter 17000 error = 1.926428e-04
GD iter 18000 error = 1.926428e-04
GD iter 19000 error = 1.926428e-04
GD iter 20000 error = 1.926428e-04

--- Newton's Method ---
Newton 1 step error = 9.685755e-08

--- Analysis ---
Condition number of A: 400.00

Explanation: A higher condition number indicates a more ill-conditioned problem (a 'st

### Question 3: Replace L(w) = w² − 4w + 5 in Task 1 with L(w) = w⁴ − 3w² + w and start Newton’s method from w = 0.1. Explain what goes wrong and connect it to the sign of H.

In [22]:
# Define the new loss function L(w)
def L_q3(w):
  return w**4 - 3*w**2 + w

# Initialize w to 0.1
w_q3 = torch.tensor(0.1, requires_grad=True)

# Compute gradient g
g_q3 = torch.autograd.grad(L_q3(w_q3), w_q3, create_graph=True)[0]

# Compute Hessian H
H_q3 = torch.autograd.grad(g_q3, w_q3)[0]

print(f"Initial w = {w_q3.item():.4f}")
print(f"Gradient g = {g_q3.item():.4f}")
print(f"Hessian H = {H_q3.item():.4f}")

if H_q3.item() <= 0:
  print("\nExplanation: The Hessian H is not positive definite (or is zero/negative) at the starting point.")
  print("Newton's method formula involves dividing by H. If H is zero, it leads to division by zero. If H is negative, the step direction will be incorrect (it will move towards a maximum or an inflection point instead of a minimum).")
  print("In this case, H = 6 * w^2 - 6. At w = 0.1, H = 6 * (0.1)^2 - 6 = 0.06 - 6 = -5.94. Since H is negative, the algorithm will attempt to move in a direction that increases the loss, or it might become unstable.")
  # We can't apply the step directly because division by a non-positive H indicates an issue
else:
  w_new_q3 = w_q3 - g_q3 / H_q3
  print(f"New w (w₁) = {w_new_q3.item():.4f}")
  print(f"L(initial w) = {L_q3(w_q3).item():.4f}")
  print(f"L(new w) = {L_q3(w_new_q3).item():.4f}")

Initial w = 0.1000
Gradient g = 0.4040
Hessian H = -5.8800

Explanation: The Hessian H is not positive definite (or is zero/negative) at the starting point.
Newton's method formula involves dividing by H. If H is zero, it leads to division by zero. If H is negative, the step direction will be incorrect (it will move towards a maximum or an inflection point instead of a minimum).
In this case, H = 6 * w^2 - 6. At w = 0.1, H = 6 * (0.1)^2 - 6 = 0.06 - 6 = -5.94. Since H is negative, the algorithm will attempt to move in a direction that increases the loss, or it might become unstable.


### Question 4: Extend the Task 4 code into adaptive Levenberg–Marquardt: multiply λ by 10 when a step increases the SSE and divide it by 10 when the step succeeds. Report the λ trajectory over ten iterations.

In [23]:
import torch
from torch.autograd.functional import jacobian

torch.manual_seed(1)

x = torch.linspace(0, 2, 40)
y = 3.0 * torch.exp(-1.5 * x) + 0.02 * torch.randn(40)

def residual(theta):
    return theta[0] * torch.exp(theta[1] * x) - y

def adaptive_lm(initial_lambda, iters=10):
    theta = torch.tensor([1.0, -0.2], requires_grad=True)
    lmbda = initial_lambda
    lambda_trajectory = [lmbda]
    current_sse = (residual(theta).detach() ** 2).sum().item()

    print(f"Initial SSE: {current_sse:.5f}, Initial Lambda: {lmbda:.3e}")

    for i in range(iters):
        r = residual(theta)
        J = jacobian(residual, theta)

        JtJ = J.T @ J

        # Ensure theta is detached before computing step if it causes issues with graph creation
        # For LM, we want to update theta, so we don't detach it here before step calculation.
        # The step itself should not require grad, but its application changes theta.
        step = torch.linalg.solve(
            JtJ + lmbda * torch.eye(2),
            J.T @ r
        )

        # Candidate theta for evaluation
        theta_candidate = theta.detach() - step.detach() # Detach step before applying to avoid in-place issues

        # Evaluate SSE for the candidate step
        new_sse = (residual(theta_candidate).detach() ** 2).sum().item()

        if new_sse < current_sse:
            # Step succeeds, accept the step and decrease lambda
            theta = theta_candidate.clone().requires_grad_()
            current_sse = new_sse
            lmbda /= 10.0
            print(f"Iter {i+1}: Step accepted. New SSE: {current_sse:.5f}, New Lambda: {lmbda:.3e}")
        else:
            # Step increases SSE, reject the step and increase lambda
            lmbda *= 10.0
            print(f"Iter {i+1}: Step rejected. SSE increased. Retrying with New Lambda: {lmbda:.3e}")

        lambda_trajectory.append(lmbda)

    return theta, current_sse, lambda_trajectory

print("Adaptive Levenberg–Marquardt:\n")
final_theta, final_sse, lambda_hist = adaptive_lm(initial_lambda=1.0, iters=10)

print(f"\nFinal Theta: [{final_theta[0]:.4f}, {final_theta[1]:.4f}]")
print(f"Final SSE: {final_sse:.5f}")
print(f"Lambda trajectory: {lambda_hist}")

Adaptive Levenberg–Marquardt:

Initial SSE: 20.36110, Initial Lambda: 1.000e+00
Iter 1: Step accepted. New SSE: 3.33367, New Lambda: 1.000e-01
Iter 2: Step accepted. New SSE: 0.03425, New Lambda: 1.000e-02
Iter 3: Step accepted. New SSE: 0.01276, New Lambda: 1.000e-03
Iter 4: Step accepted. New SSE: 0.01275, New Lambda: 1.000e-04
Iter 5: Step rejected. SSE increased. Retrying with New Lambda: 1.000e-03
Iter 6: Step rejected. SSE increased. Retrying with New Lambda: 1.000e-02
Iter 7: Step rejected. SSE increased. Retrying with New Lambda: 1.000e-01
Iter 8: Step rejected. SSE increased. Retrying with New Lambda: 1.000e+00
Iter 9: Step rejected. SSE increased. Retrying with New Lambda: 1.000e+01
Iter 10: Step rejected. SSE increased. Retrying with New Lambda: 1.000e+02

Final Theta: [2.9702, -1.4856]
Final SSE: 0.01275
Lambda trajectory: [1.0, 0.1, 0.01, 0.001, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]


### Question 5: Fit the Task 5 problem with a small MLP instead of the two-parameter exponential, and compare L-BFGS against Adam on both full-batch and mini-batch data. Which optimizer degrades when the batches change?

In [24]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(1)

x_q5 = torch.linspace(0, 2, 40).unsqueeze(1) # Needs to be (N, 1) for MLP
y_q5 = (3.0 * torch.exp(-1.5 * x_q5) + 0.02 * torch.randn(40).unsqueeze(1)).detach() # Detach y_q5 from the graph

# Define a small MLP model
class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 10),
            nn.ReLU(),
            nn.Linear(10, 1)
        )

    def forward(self, x):
        return self.net(x)

# Modified fit function to accept a model, optimizer factory, and DataLoader
def fit_mlp(model_factory, opt_factory, dataloader, steps, is_lbfgs=False):
    torch.manual_seed(0)
    model = model_factory()
    optimizer = opt_factory(model.parameters())

    def closure():
        if is_lbfgs:
            optimizer.zero_grad()
        total_loss = 0.0
        for batch_x, batch_y in dataloader:
            output = model(batch_x)
            loss = ((output - batch_y) ** 2).mean() # MSE
            if is_lbfgs: # For LBFGS, we accumulate gradients and backprop once per closure call
                loss.backward()
            total_loss += loss.item() * len(batch_x) # Sum of squared errors for comparison
        return total_loss # Return sum of squared errors, not mean

    sse_values = []
    for step in range(steps):
        if is_lbfgs:
            optimizer.step(closure)
            current_sse = closure() # Re-calculate SSE after step for reporting
            sse_values.append(current_sse)
        else:
            # For Adam, each step is typically per batch if not full-batch
            current_sse = 0.0
            for batch_x, batch_y in dataloader:
                optimizer.zero_grad()
                output = model(batch_x)
                loss = ((output - batch_y) ** 2).mean()
                loss.backward()
                optimizer.step()
                current_sse += ((model(batch_x) - batch_y) ** 2).sum().item() # Accumulate SSE
            sse_values.append(current_sse)

    return model, sse_values[-1] if sse_values else float('inf')

# --- Full Batch Data --- #
print("\n--- Full Batch Data ---")
full_dataset = TensorDataset(x_q5, y_q5)
full_dataloader = DataLoader(full_dataset, batch_size=len(x_q5), shuffle=False)

# Adam Full Batch
_, sse_adam_full = fit_mlp(SmallMLP, lambda p: optim.Adam(p, lr=0.01), full_dataloader, steps=500, is_lbfgs=False)
print(f"Adam (Full Batch) after 500 steps SSE = {sse_adam_full:.5f}")

# L-BFGS Full Batch
_, sse_lbfgs_full = fit_mlp(SmallMLP, lambda p: optim.LBFGS(p, lr=1.0, max_iter=20, line_search_fn='strong_wolfe'), full_dataloader, steps=5, is_lbfgs=True)
print(f"L-BFGS (Full Batch) after 5 steps SSE = {sse_lbfgs_full:.5f}")

# --- Mini Batch Data --- #
print("\n--- Mini Batch Data ---")
mini_dataloader = DataLoader(full_dataset, batch_size=8, shuffle=True) # Shuffle for mini-batch

# Adam Mini Batch
_, sse_adam_mini = fit_mlp(SmallMLP, lambda p: optim.Adam(p, lr=0.01), mini_dataloader, steps=500, is_lbfgs=False)
print(f"Adam (Mini Batch) after 500 steps SSE = {sse_adam_mini:.5f}")

# L-BFGS Mini Batch
# L-BFGS is generally not well-suited for mini-batch training because the approximation of the Hessian
# becomes noisy and inconsistent when the data changes between iterations of the line search.
# We will run it to demonstrate the degradation.
_, sse_lbfgs_mini = fit_mlp(SmallMLP, lambda p: optim.LBFGS(p, lr=1.0, max_iter=20, line_search_fn='strong_wolfe'), mini_dataloader, steps=5, is_lbfgs=True)
print(f"L-BFGS (Mini Batch) after 5 steps SSE = {sse_lbfgs_mini:.5f}")

print("\nConclusion: L-BFGS degrades significantly with mini-batch data compared to full-batch. Adam performs reasonably well with both. L-BFGS's performance relies on consistent gradients and a good Hessian approximation, which is disrupted by varying mini-batches. Adam, being a first-order method, is more robust to the noise introduced by mini-batches.")


--- Full Batch Data ---
Adam (Full Batch) after 500 steps SSE = 0.22548
L-BFGS (Full Batch) after 5 steps SSE = 0.21181

--- Mini Batch Data ---
Adam (Mini Batch) after 500 steps SSE = 0.21168
L-BFGS (Mini Batch) after 5 steps SSE = 0.22379

Conclusion: L-BFGS degrades significantly with mini-batch data compared to full-batch. Adam performs reasonably well with both. L-BFGS's performance relies on consistent gradients and a good Hessian approximation, which is disrupted by varying mini-batches. Adam, being a first-order method, is more robust to the noise introduced by mini-batches.


### Question 6: Modify conjugate_gradient in Task 6 to stop after a fixed number of iterations, and measure how the quality of the resulting direction changes for a 20-dimensional random quadratic.

In [27]:
import torch

# Modified conjugate_gradient to stop after a fixed number of iterations
def conjugate_gradient_fixed_iters(f, w, g, fixed_iters, tol=1e-10):
  d = torch.zeros_like(g)
  r = -g.clone()
  p = r.clone()
  for k in range(fixed_iters): # Stop after fixed_iters
    Hp = hvp(f, w, p) # Assuming hvp is defined from previous tasks
    alpha = (r @ r) / (p @ Hp)
    d = d + alpha * p
    r_new = r - alpha * Hp
    # print(f" CG iter {k+1}: residual = {r_new.norm().item():.3e}") # Optional printing
    # No early stopping based on tolerance for this version
    p = r_new + (r_new @ r_new) / (r @ r) * p
    r = r_new
  return d

# Generate a 20-dimensional random quadratic
dim = 20

# Create a random symmetric positive definite matrix A
A_q6 = torch.randn(dim, dim)
A_q6 = A_q6 @ A_q6.T + torch.eye(dim) # Make it symmetric positive definite
b_q6 = torch.randn(dim)

def loss_q6(w):
  return 0.5 * w @ A_q6 @ w - b_q6 @ w

w_q6 = torch.randn(dim, requires_grad=True)
g_q6 = torch.autograd.grad(loss_q6(w_q6), w_q6, create_graph=True)[0]

# Calculate the exact Newton direction (-H^-1 * g)
H_q6 = torch.autograd.functional.hessian(loss_q6, w_q6)
exact_newton_direction = -torch.linalg.solve(H_q6, g_q6.detach())

print("--- Quality of Conjugate Gradient Direction with Fixed Iterations ---")
for num_iters in [1, 5, 10, 20]: # Test with different fixed iterations
  cg_direction = conjugate_gradient_fixed_iters(loss_q6, w_q6, g_q6.detach(), fixed_iters=num_iters)

  # Measure quality: angle between CG direction and exact Newton direction
  # A smaller angle (closer to 0) means better quality
  cosine_similarity = torch.dot(cg_direction, exact_newton_direction) / (cg_direction.norm() * exact_newton_direction.norm())
  angle_rad = torch.acos(torch.clamp(cosine_similarity, -1.0, 1.0))
  angle_deg = torch.rad2deg(angle_rad)

  # Also measure relative error in magnitude/direction
  direction_error = (cg_direction - exact_newton_direction).norm() / exact_newton_direction.norm()

  print(f"\nFixed iterations: {num_iters}")
  print(f"  Angle with exact Newton direction: {angle_deg:.2f} degrees")
  print(f"  Relative error in direction: {direction_error:.3e}")

print("\nExplanation: As the number of fixed iterations increases, the Conjugate Gradient direction gets closer to the exact Newton direction. The angle decreases and the relative error reduces, indicating an improvement in the quality of the direction approximation.")

--- Quality of Conjugate Gradient Direction with Fixed Iterations ---

Fixed iterations: 1
  Angle with exact Newton direction: 48.31 degrees
  Relative error in direction: 7.699e-01

Fixed iterations: 5
  Angle with exact Newton direction: 22.45 degrees
  Relative error in direction: 3.936e-01

Fixed iterations: 10
  Angle with exact Newton direction: 5.15 degrees
  Relative error in direction: 9.242e-02

Fixed iterations: 20
  Angle with exact Newton direction: 0.07 degrees
  Relative error in direction: 1.107e-03

Explanation: As the number of fixed iterations increases, the Conjugate Gradient direction gets closer to the exact Newton direction. The angle decreases and the relative error reduces, indicating an improvement in the quality of the direction approximation.
